# Omni NCM — first clean architecture comparison: 82.9M params, 167k unique samples

**Read this first.** The 10M numbers from earlier runs are **not usable**, and neither is the
"more data did not help" conclusion. `ShardStream` walked `files[0]` on every pass and never
advanced, so both the 4k and the 11k run trained on the **same single shard** — about 60 rows.
Held-out accuracy was flat between them because the two runs were, data-wise, the same experiment.
Fixed and regression-tested (`tests/test_shard_stream.py`); cell 1 pulls the fix.

That makes this run the first honest comparison in the project. It trains **82.88M params** on
**167k unique samples** with the same schedule, vocab and mask machinery, and only the mask and
input shift differing between arms.

| | this run |
|---|---:|
| params | **82.88M** (d768, L13, h12, kv3) |
| packed positions/sample | 504 |
| unique samples | **167,000** |
| positions per parameter | **1.0** |
| eval rows (text/image/audio/video) | **128/96/64/32** |
| arms | `strict` (level-parallel) vs `ar` (matched-params autoregressive) |

**Cost on a Colab A100:** production ~1 h at the measured 45 samples/s (8 workers), training
~20 min per arm, eval ~5 min. Inside your credits; no rental.

**Decision rule, fixed before the run.** Margin = `strict - AR` at text level 5.
`< 0.05` kill · `0.05-0.10` weak, report and stop · `>= 0.10` holds, and that is what justifies
the 315M rung.

**Why not `base_len=1024`.** At 1024 the packed sequence is 7,119 positions and attention needs
`B x H x T x T` scores — 9 GiB for the block mask alone at batch 2, and the eager fallback runs
out of memory before a step completes. Long sequence here is a chunked-attention job, not a config
change.


In [ ]:
import os, sys, subprocess

REPO = 'adi274903/ncm-omni'          # private
PKG  = os.environ.get('NCM_PKG', '/content/ncm_omni')

def _token():
    """Read-only clone access. Colab secret first, then env, then an authenticated gh."""
    try:
        from google.colab import userdata
        t = userdata.get('GITHUB_TOKEN')
        if t:
            return t.strip()
    except Exception:
        pass
    t = os.environ.get('GITHUB_TOKEN')
    if t:
        return t.strip()
    try:
        return subprocess.check_output(['gh', 'auth', 'token'], text=True).strip()
    except Exception:
        return None

def _askpass(tok):
    """Hand the token to git through GIT_ASKPASS, never through argv or .git/config.

    The old cell pulled with the plain https URL, but the clone deliberately stripped the
    credential out of origin - so a re-run died with `could not read Username`. The token has to
    be re-supplied on every pull, and it must not land in the notebook output or on disk.
    """
    d = os.path.dirname(os.path.abspath(PKG))
    secret = os.path.join(d, '.git-tok')
    with open(secret, 'w') as f:
        f.write(tok)
    os.chmod(secret, 0o600)
    p = os.path.join(d, '.git-askpass.sh')
    with open(p, 'w') as f:
        f.write('#!/bin/sh\ncase "$1" in\n  *sername*) echo x-access-token ;;\n'
                '  *) cat "{}" ;;\nesac\n'.format(secret))
    os.chmod(p, 0o700)
    return p, secret

def _git(args, tok=None):
    env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
    junk = []
    if tok:
        ap, secret = _askpass(tok)
        junk = [ap, secret]
        env.update(GIT_ASKPASS=ap, GIT_ASKPASS_REQUIRE='force')
    try:
        # credential.helper= clears any stale helper so askpass is the only source
        return subprocess.check_output(['git', '-c', 'credential.helper='] + args,
                                       text=True, stderr=subprocess.STDOUT, env=env).strip()
    finally:
        for f in junk:
            if os.path.exists(f):
                os.remove(f)

assert PKG.startswith('/content') or os.environ.get('NCM_PKG'), 'refusing to touch ' + PKG
tok = _token()
url = 'https://github.com/%s.git' % REPO
if os.path.isdir(os.path.join(PKG, '.git')):
    try:
        # Fetch + hard reset, not pull: the checkout is a --depth 1 graft, so a rewound history
        # cannot fast-forward, and run/cfg is rewritten by every invocation. The repo holds no
        # edits worth keeping (run/ is gitignored), so landing exactly on origin/main is right.
        _git(['-C', PKG, 'fetch', '--depth', '1', 'origin', 'main'], tok)
        _git(['-C', PKG, 'reset', '--hard', 'FETCH_HEAD'], tok)
    except subprocess.CalledProcessError as e:
        # Re-clone, but keep run/ (shards + checkpoints). A plain `rm -rf` here silently deletes
        # hours of training because run/ is gitignored and therefore not in the repo.
        print('update failed -> re-cloning (keeping run/):',
              str(e.output).strip().splitlines()[-1])
        import shutil
        tmp = PKG.rstrip('/') + '.new'
        shutil.rmtree(tmp, ignore_errors=True)
        _git(['clone', '--depth', '1', url, tmp], tok)
        keep = os.path.join(PKG, 'run')
        if os.path.isdir(keep):
            shutil.rmtree(os.path.join(tmp, 'run'), ignore_errors=True)
            shutil.move(keep, os.path.join(tmp, 'run'))
        shutil.rmtree(PKG, ignore_errors=True)
        os.rename(tmp, PKG)
else:
    _git(['clone', '--depth', '1', url, PKG], tok)

assert os.path.exists(os.path.join(PKG, 'core.py')), 'clone landed without core.py'
os.chdir(PKG)
if os.environ.get('NCM_SKIP_PIP') != '1':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                           'datasets', 'soundfile', 'av'])

cfg = os.path.join(PKG, '.git', 'config')
assert 'x-access-token' not in open(cfg).read(), 'credential leaked into .git/config'

import torch
print('repo   ', REPO, '| commit', subprocess.check_output(
    ['git', '-C', PKG, 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('torch  ', torch.__version__, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('workdir', os.getcwd())


In [ ]:
# HF token + hardware probe. Unauthenticated Hub requests answer 429 on bursts of stream opens,
# which silently drops sources and changes the mix. The token is read from Colab secrets and
# never printed.
import os, shutil, subprocess, sys

def _hf_token():
    try:
        from google.colab import userdata
        for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            try:
                t = userdata.get(k)
                if t:
                    return t.strip()
            except Exception:
                pass
    except Exception:
        pass
    for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k):
            return os.environ[k].strip()
    return None

t = _hf_token()
if t:
    os.environ["HF_TOKEN"] = t           # picked up by huggingface_hub in-process and by children
    print("HF token: loaded (%d chars)" % len(t))
else:
    print("HF token: MISSING -> add a Colab secret named HF_TOKEN "
          "(key icon, left sidebar). Without it the producer will 429 and skip sources.")

try:
    from huggingface_hub import whoami
    print("HF auth: ", whoami()["name"] if os.environ.get("HF_TOKEN") else "anonymous")
except Exception as e:
    print("HF auth check skipped:", type(e).__name__)

cpu = os.cpu_count()
ram = shutil.disk_usage("/").total / 1e9
try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0
except Exception:
    gpu, vram = "unknown", 0
def _ram_gb():
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemTotal"):
                return int(line.split()[1]) / 1e6
    except Exception:
        pass
    return 0.0
print(f"cpu {cpu} threads | RAM {_ram_gb():.0f} GB | disk {ram:.0f} GB | {gpu} ({vram} GB)")
print("WORKERS note: 8 measured at 45 samples/s on a 12-vCPU runtime with a token and a clean process table.")


In [ ]:
# Invariants at the configured schedule. Must print PIPELINE CHECK OK.
!python run_omni.py check 2>&1 | tail -8

In [ ]:
# ---- CONFIG: the only cell you edit ------------------------------------------------
PRESET    = "md"        # md = 82.88M (d768 L13 h12 kv3). "sm" = the 10.44M debug harness, "lg" = 454M
BASE_LEN  = 64          # 504 packed positions/sample. 1024 => 7,119 positions and a memory wall
BUDGET    = 167000      # unique samples to produce
HOLDOUT   = 4           # x4 => 128 text / 96 image / 64 audio / 32 video eval rows (SE ~0.028)
BATCH     = 16          # A100 40GB holds this at 504 positions; drop to 8 if you see OOM
WORKERS   = 8           # 8 measured at 45 samples/s on a 12-vCPU runtime
TPP       = 1.0         # packed positions per parameter. 0.11 = the 10M run; 1.0 = this run

PARAMS_MD = 82.88e6     # printed by the train cell as "params 82.88M" - correct it if it differs
POS_PER_SAMPLE = 504
STEPS = int(TPP * PARAMS_MD / (BATCH * POS_PER_SAMPLE))
print(f"preset {PRESET} | base_len {BASE_LEN} | {POS_PER_SAMPLE} pos/sample | budget {BUDGET} | "
      f"holdout x{HOLDOUT} | batch {BATCH} | {STEPS:,} steps at {TPP} pos/param")
print(f"expected production time at 45 samples/s: {BUDGET/45/60:.0f} min")


In [ ]:
# ---- PRODUCE: 167k unique samples ---------------------------------------------------
# Shard contents depend on base_len and there is no version stamp in the filename, so an old
# stream would be trained against the wrong schedule. Wipe it.
import glob, json, os, shutil, subprocess, sys, time

STREAM, SHARD, LOG = "run/stream", 64, "run/produce.log"

def shards():
    return len(glob.glob(STREAM + "/shard_*.npz"))

def done():
    return os.path.exists(os.path.join(STREAM, "DONE"))

def stats():
    """True progress. Shard count only ticks every 64 samples per worker, so it reads 0 for the
    first minute even when every worker is pulling rows - the per-worker progress files do not."""
    tot, per, held = 0, {}, {}
    for f in sorted(glob.glob(STREAM + "/progress_w*.json")) + glob.glob(STREAM + "/progress.json"):
        try:
            j = json.load(open(f))
        except Exception:
            continue
        tot += j.get("samples", 0)
        for k, v in (j.get("per_source") or {}).items():
            per[k] = per.get(k, 0) + v
        for g, v in (j.get("eval_held") or {}).items():
            held[g] = max(held.get(g, 0), v)
    return tot, per, held

if os.path.isdir(STREAM) and not done():
    print(f"incomplete stream ({shards()} shards) from another config - removing")
    shutil.rmtree(STREAM)
os.makedirs(STREAM, exist_ok=True)

if done() and shards() >= BUDGET // SHARD - 8:
    print("stream complete:", shards(), "shards - skipping production")
else:
    p = subprocess.Popen([sys.executable, "-u", "run_omni.py", "produce",
                          "--budget", str(BUDGET), "--workers", str(WORKERS),
                          "--preset", PRESET, "--base-len", str(BASE_LEN),
                          "--holdout-mult", str(HOLDOUT), "--max-ahead", "1000000",
                          "--foreground"],
                         stdout=open(LOG, "a"), stderr=subprocess.STDOUT, start_new_session=True)
    print(f"producer pid {p.pid} | {BUDGET} samples | {WORKERS} workers -> {LOG}")
    print(f"zero shards is normal for the first {512 + SHARD} samples per worker: each worker fills "
          f"a 512-sample shuffle buffer and then a {SHARD}-sample shard before the first file lands.")
    t0, tick, quiet, prev = time.time(), 0.0, time.time(), 0
    while True:
        time.sleep(20)
        n, el = shards(), time.time() - t0
        smp, per, held = stats()
        if smp != prev:
            prev, quiet = smp, time.time()
        if el - tick >= 60 or done():
            rate = smp / el if el else 0.0
            eta = (BUDGET - smp) / rate / 60 if rate > 0.05 else float("nan")
            hold = " ".join(f"{g}:{v}" for g, v in held.items()) or "-"
            rows = sum(per.values())
            print(f"[{el/60:5.1f} min] {n:5d} shards | {smp:6d}/{BUDGET} samples "
                  f"({rows} rows decoded) | {rate:5.2f}/s | ETA {eta:5.0f} min "
                  f"| sources {len(per):2d} | holdout {hold}", flush=True)
            tick = el
        if done():
            break
        if p.poll() is not None:
            print("producer exited before DONE. Last lines:")
            print("".join(open(LOG).readlines()[-20:]))
            break
        if time.time() - quiet > 420:
            print("\nno new samples in 7 min. Last lines of the aggregate log:")
            print("".join(open(LOG).readlines()[-15:]) if os.path.exists(LOG) else "(no log)")
            wl = sorted(glob.glob("run/producer_w*.log"))
            for f in wl[:3]:
                print(f"-- {f}: " + "".join(open(f).readlines()[-2:]).strip()[:200])
            break
    print("shards:", shards(), "| EVAL_READY:", os.path.exists(STREAM + "/EVAL_READY"), "| DONE:", done())


In [ ]:
# ---- TRAIN: strict arm, 82.9M ----------------------------------------------------
# Scratch the 10M checkpoints: eval reads every arm listed in run/train_*, so a leftover
# 10.44M checkpoint would be compared against an 82.9M model and reported as a result.
!rm -rf run/train_strict run/train_ar run/train_block
!python run_omni.py train --arm strict --preset md --base-len 64 --batch 16 --steps 10278 2>&1 | tee run/train_strict.log | tail -40


In [ ]:
# ---- TRAIN: AR arm, same params, same tokens, same schedule -----------------------
!python run_omni.py train --arm ar --preset md --base-len 64 --batch 16 --steps {STEPS} 2>&1 | tee run/train_ar.log | tail -40


In [ ]:
# ---- EVAL: 320 held-out rows, conditional table only -----------------------------
!python run_omni.py eval --preset md --base-len 64 --holdout-mult 4 2>&1 | tee run/eval.log | tail -45


In [ ]:
# ---- VERDICT: the level-parallel margin, on real data ------------------------------
import json, os

tab = json.load(open("run/eval_table.json"))
cond = tab["conditional"]
st = cond.get("level-parallel(strict)", {})
ar = cond.get("autoregressive", {})
floor = tab["baseline_by_level"].get("5", tab["baseline"])

print(f"{'level':>6} | {'strict':>7} | {'AR':>7} | {'margin':>7}")
for k in range(6):
    key = f"text_L{k}"
    if key in st:
        a = ar.get(key, float("nan"))
        print(f"{k:>6} | {st[key]:7.3f} | {a:7.3f} | {st[key] - a:+7.3f}")
print(f"\nconstant floor, text L5: {floor:.3f}")
for m in ("image", "audio", "video"):
    if f"{m}_L5" in st:
        print(f"{m:>7} L5: strict {st[f'{m}_L5']:.3f} | AR {ar.get(f'{m}_L5', float('nan')):.3f}")

margin = st.get("text_L5", float("nan")) - ar.get("text_L5", float("nan"))
print(f"\nmargin at text L5: {margin:+.3f}")
print()
if margin != margin:
    print("VERDICT: no margin - an arm is missing from run/train_*. Re-run the missing train cell.")
elif st.get("text_L5", 0) <= floor:
    print("VERDICT: KILL. strict does not beat the constant floor. The level-parallel claim fails "
          "at 82.9M on real data.")
elif margin < 0.05:
    print("VERDICT: KILL. No usable advantage over a matched-params AR baseline at this scale.")
elif margin < 0.10:
    print("VERDICT: WEAK. A real but small margin. Report it, do not spend on 315M yet - raise "
          "positions per parameter first and see if it widens.")
else:
    print("VERDICT: HOLDS. strict beats AR by >=0.10 at 82.9M on 167k unique samples. That is the "
          "justification for the 315M run, which needs chunked attention first.")

for d in ("/content/drive/MyDrive/ncm", "/content", "."):
    try:
        os.makedirs(d, exist_ok=True)
        dst = os.path.join(d, "eval_table_killtest.json")
        json.dump(tab, open(dst, "w"), indent=2)
        print("\nsaved", dst)
        break
    except Exception:
        continue


## Reading the result

- **The margin, not the absolute number.** Absolute accuracy depends on the eval set, which just
  changed size (32 -> 128 text rows), so 0.519 is no longer directly comparable. The margin is.
- **Per modality, never the mean.** Video was dead at 10M (0.129 vs 0.183 floor) and audio barely
  cleared its floor; if they are still below the constant, say so rather than averaging them in.
- **Free-running stays void.** Level 0 is a 1-of-256 guess; eval labels any margin under 0.05 a tie.
- **This is the first clean run.** Anything quoted from the 4k/11k stream predates the
  `ShardStream` fix and trained on a single shard; do not put those numbers in a paper.
- **1 position per parameter is not Chinchilla.** Both arms are under-trained by design — the run
  is long enough to compare architectures, not to produce a good model. Say that in the paper.

## If something breaks

| symptom | cause | fix |
|---|---|---|
| `CUDA out of memory` in train | batch too large for the card | `--batch 8`, or `BATCH = 8` and re-run cell 4's step count |
| produce stops growing, no error | Hub throttling, or a sharded read is stuck | cell 4 self-stops after 7 min; check `tail run/produce.log` |
| eval prints `VOID` everywhere | level-0 free-running is a coin flip | expected; only the conditional table carries the claim |
| a `LEGACY-leaky` row appears in eval | an old `train_block` checkpoint survived | `rm -rf run/train_block` and re-run eval |
